# Task 3 — Core Architecture Demo
**PlaceMux AI/ML Phase 1** | Config-driven modular train/eval harness


In [1]:
import sys, os
sys.path.insert(0, os.path.join('..'))
import yaml, pandas as pd
from src.data.loader import load_data, split_data
from src.pipeline import build_pipeline
from src.evaluation.evaluate import evaluate, log_metrics
print('✅ All modules imported')

✅ All modules imported


## Step 1 — Load Config

In [2]:
with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)
print('Config loaded:')
print(yaml.dump(cfg, default_flow_style=False))

Config loaded:
data:
  drop_cols: []
  path: data/credit_fraud_dataset.csv
  target_col: is_fraud
experiment:
  log_path: experiments/metrics.csv
model:
  name: dummy
  params:
    strategy: most_frequent
training:
  random_seed: 42
  test_size: 0.15
  val_size: 0.15



## Step 2 — Load & Split Data

In [3]:
X, y = load_data('../' + cfg['data']['path'], target_col=cfg['data']['target_col'])
X_tr, X_val, X_test, y_tr, y_val, y_test = split_data(
    X, y, val_size=cfg['training']['val_size'],
    test_size=cfg['training']['test_size'], random_seed=cfg['training']['random_seed'])
print(f'Train:{len(X_tr)} | Val:{len(X_val)} | Test:{len(X_test)}')

  [DataLoader] Loaded 1500 rows, 13 features. Target 'is_fraud' balance: {0: 1058, 1: 442}
  [DataLoader] Split → Train:1050 | Val:225 | Test:225
Train:1050 | Val:225 | Test:225


## Step 3 — Run Dummy Baseline (default config)

In [4]:
pipe = build_pipeline(X_tr, 'dummy', {'strategy':'most_frequent'}, 42)
pipe.fit(X_tr, y_tr)
m = evaluate(pipe, X_val, y_val, 'val')
print('Dummy val F1:', m['f1_macro'])

  [Preprocessor] Numeric: ['age', 'income', 'credit_limit', 'transaction_amount', 'num_transactions_30d', 'account_age_months', 'num_prev_disputes', 'country_match', 'time_of_day_hour', 'is_weekend', 'card_present', 'distance_from_home_km'] | Categorical: ['merchant_category']
  [ModelFactory] Created 'dummy' → DummyClassifier(random_state=42, strategy='most_frequent')
  [Pipeline] Built: Preprocessor → dummy

  [Evaluator] VAL metrics:
    accuracy    : 0.7022
    precision   : 0.3511
    recall      : 0.5
    f1_macro    : 0.4125

              precision    recall  f1-score   support

   Not Fraud       0.70      1.00      0.83       158
       Fraud       0.00      0.00      0.00        67

    accuracy                           0.70       225
   macro avg       0.35      0.50      0.41       225
weighted avg       0.49      0.70      0.58       225

Dummy val F1: 0.4125


## Step 4 — Swap to Logistic Regression (ONE line change)

In [5]:
# Only the model name changes — data, eval, logging untouched
pipe_lr = build_pipeline(X_tr, 'logistic', {}, 42)
pipe_lr.fit(X_tr, y_tr)
m_lr = evaluate(pipe_lr, X_val, y_val, 'val')
print('Logistic val F1:', m_lr['f1_macro'])

  [Preprocessor] Numeric: ['age', 'income', 'credit_limit', 'transaction_amount', 'num_transactions_30d', 'account_age_months', 'num_prev_disputes', 'country_match', 'time_of_day_hour', 'is_weekend', 'card_present', 'distance_from_home_km'] | Categorical: ['merchant_category']
  [ModelFactory] Created 'logistic' → LogisticRegression(random_state=42)
  [Pipeline] Built: Preprocessor → logistic

  [Evaluator] VAL metrics:
    accuracy    : 0.7778
    precision   : 0.7353
    recall      : 0.7128
    f1_macro    : 0.7217

              precision    recall  f1-score   support

   Not Fraud       0.82      0.87      0.85       158
       Fraud       0.65      0.55      0.60        67

    accuracy                           0.78       225
   macro avg       0.74      0.71      0.72       225
weighted avg       0.77      0.78      0.77       225

Logistic val F1: 0.7217


## Step 5 — View Experiment Log

In [6]:
pd.read_csv('../experiments/metrics.csv')[['timestamp','model','accuracy','f1_macro','split']]

,timestamp,model,accuracy,f1_macro,split
0,2026-08-15 05:18:45,dummy,0.7022,0.4125,val
1,2026-08-15 05:18:45,dummy,0.7067,0.4141,test
2,2026-08-15 05:18:49,logistic,0.7778,0.7217,val
3,2026-08-15 05:18:49,logistic,0.7778,0.7063,test
4,2026-08-15 05:18:53,random_forest,0.8000,0.7483,val
5,2026-08-15 05:18:53,random_forest,0.8000,0.7401,test
6,2026-08-15 10:57:48,dummy,0.7022,0.4125,val
7,2026-08-15 10:57:48,dummy,0.7067,0.4141,test


## ✅ Architecture Checklist

| Item | Status |
|---|---|
| Modular data/features/model/eval | ✅ |
| YAML config controls all params | ✅ |
| sklearn Pipeline | ✅ |
| Single harness (train.py) | ✅ |
| Dummy baseline wired through | ✅ |
| Model swap without rewrites | ✅ |
| Metrics auto-logged to CSV | ✅ |
| Real dataset (1500 rows) | ✅ |